# Ablation: what made Custom CNN v2 better?

Custom CNN v2 beat the v1 baseline by ~9 macro-F1 points, but it changed **two
things at once**: the architecture (residual blocks + SE attention) and the
training recipe (AdamW, cosine warmup, label smoothing, class-balanced focal
loss, MixUp, CutMix). The thesis cannot currently say which one earned the gain,
and it separately claims that *augmentation substitutes for pretraining on the
robustness axis* on mechanism alone, with no measurement behind it.

This notebook runs the 2x2 that settles both questions.

| Run | Architecture | Recipe | Answers |
|-----|--------------|--------|---------|
| A `arch_only` | v2 (residual+SE) | plain (Adam, CE) | gain from architecture alone |
| B `recipe_only` | v1 (plain CNN) | full Tier-1 | gain from recipe alone |
| C `aug_only` | v1 (plain CNN) | MixUp+CutMix **only** | **does augmentation alone close the lab->field gap?** |

**Run C is the decisive one.** It isolates augmentation, which is the only claim
in the thesis currently resting on argument rather than data. If you can afford
exactly one run, set `RUNS = ["aug_only"]`.

---

## Read this before attaching anything

> ### This notebook TRAINS. It cannot reuse your existing `.pth` files.
>
> Attaching the trained model datasets does nothing here - unlike
> `Kaggle_Run_From_Git.ipynb`, this notebook has no `MODE = "analyze"` path and
> never reads `/kaggle/input` for weights.
>
> That is not an oversight. A checkpoint records **one** architecture trained under
> **one** recipe, and the two you already have are the endpoints of this experiment:
>
> | Existing checkpoint | What it already is |
> |---|---|
> | `custom_cnn` (v1) | v1 architecture + plain recipe = the **baseline** endpoint |
> | `custom_cnn_v2` | v2 architecture + full Tier-1 = the **full** endpoint |
>
> The three ablation cells need combinations that were **never trained** - v1 with
> augmentation only, v1 with the full recipe, v2 with the plain recipe. No
> post-hoc analysis can recover them from the two endpoints, so each must be
> trained. Only the endpoints are reused, and they are read from the constants in
> the attribution cell rather than retrained.

Attach only the **image dataset**.

---

### How to use

1. Attach the image dataset.
2. Set `SMOKE = False` and pick `RUNS` in the config cell.
3. Run every cell top to bottom.
4. Download `ablation_bundle.zip`.

**Budget:** roughly 5-7 GPU-hours per run at 50 epochs, so all three total ~17 h -
more than one Kaggle session allows (12 h). Run **one per session**; each writes its
own output directory, and a finished run is skipped if you re-run the cell.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BDT-17/Internship.git"
REPO_DIR = Path("/kaggle/working/Internship")
BRANCH = None

# --- Which ablation runs to execute ---------------------------------------
# "aug_only"    (C) augmentation only, v1 architecture  <- the decisive one
# "recipe_only" (B) full Tier-1 recipe, v1 architecture
# "arch_only"   (A) v2 architecture, plain recipe
#
# One run per session: a 50-epoch run costs ~5-7 GPU-hours and Kaggle cuts the
# session at 12 h, so queueing all three loses whichever is in flight.
# Ordered so the most informative run comes first.
RUNS = ["aug_only"]
# RUNS = ["recipe_only"]   # session 2
# RUNS = ["arch_only"]     # session 3

SMOKE = False    # True -> 1 epoch, pipeline check only. Set False for real numbers.
EPOCHS = 50      # must match the 50-epoch budget of the runs already reported
BATCH_SIZE = 16  # must match: the reported runs used 16, not the code default 32
SEED = 42        # same split as every reported run

OUTPUT_ROOT = Path("/kaggle/working/ablation")

# Measured from the 1-epoch smoke run; used only to warn before a long session.
MINUTES_PER_EPOCH = {"aug_only": 8.1, "recipe_only": 6.2, "arch_only": 6.0}

if SMOKE:
    print("!! SMOKE=True -> 1 epoch per run. Pipeline check, NOT results.")
    print("   The analysis, table and bundle cells will refuse to run.")
    print("   Set SMOKE=False before generating anything you will cite.")
else:
    hours = sum(MINUTES_PER_EPOCH.get(r, 7.0) for r in RUNS) * EPOCHS / 60
    print(f"Real run: {EPOCHS} epochs x {len(RUNS)} run(s), batch {BATCH_SIZE}, seed {SEED}")
    print(f"Estimated GPU time: ~{hours:.1f} h")
    if hours > 11:
        print(f"   !! ~{hours:.1f} h exceeds the 12 h Kaggle session limit.")
        print("      Run fewer runs per session - a cut-off run is lost work.")
print("Runs queued:", ", ".join(RUNS))

In [ ]:
import subprocess

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "fetch", "--all"], cwd=REPO_DIR, check=True)
    if BRANCH:
        subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)
else:
    command = ["git", "clone"]
    if BRANCH:
        command.extend(["--branch", BRANCH])
    command.extend([REPO_URL, str(REPO_DIR)])
    subprocess.run(command, check=True)

print("Repository ready:", REPO_DIR)

## The ablation grid

Every run below shares the dataset, the seed-42 split, the epoch budget and the
batch size with the four models already in the thesis. **Only the factor under
test changes** - that is the whole point, and it is why the flags are written out
explicitly rather than inherited from a config.

`no_pretrained` is irrelevant here (both architectures train from scratch), and
every run keeps `--model-selection-metric val_f1_macro` so the checkpoint choice
matches the reported runs.

In [ ]:
# --- Definition of each ablation cell in the 2x2 -------------------------------
# Baseline v1 recipe = Adam, no scheduler, no label smoothing, plain CE, no mixing.
# Full Tier-1 recipe = AdamW + cosine warmup + label smoothing + cb_focal + mixup/cutmix.
# Each dict below is the DIFF from the plain baseline.

PLAIN = {
    "optimizer": "adam",
    "lr_scheduler": "none",
    "label_smoothing": 0.0,
    "loss": "ce",
    "mixup_alpha": 0.0,
    "cutmix_alpha": 0.0,
}
TIER1 = {
    "optimizer": "adamw",
    "lr_scheduler": "cosine_warmup",
    "label_smoothing": 0.1,
    "loss": "cb_focal",
    "mixup_alpha": 0.2,
    "cutmix_alpha": 1.0,
}
# Augmentation in isolation: mixing on, everything else at baseline.
AUG_ONLY = dict(PLAIN, mixup_alpha=0.2, cutmix_alpha=1.0)

ABLATIONS = {
    "aug_only": {
        "model": "custom_cnn",
        "recipe": AUG_ONLY,
        "question": "Does augmentation ALONE narrow the lab->field gap?",
    },
    "recipe_only": {
        "model": "custom_cnn",
        "recipe": TIER1,
        "question": "How much of the +9 macro-F1 comes from the recipe alone?",
    },
    "arch_only": {
        "model": "custom_cnn_v2",
        "recipe": PLAIN,
        "question": "How much comes from residual+SE alone?",
    },
}

for name in RUNS:
    spec = ABLATIONS[name]
    changed = {k: v for k, v in spec["recipe"].items() if PLAIN[k] != v} or "none (plain baseline)"
    print(f"{name:12s} arch={spec['model']:14s} changed={changed}")
    print(f"{'':12s} -> {spec['question']}\n")

In [ ]:
# --- Run the ablations -------------------------------------------------------
# One train_kaggle.py invocation per run, each into its own --output-dir so a
# crashed or timed-out run cannot corrupt a finished one. Re-running the cell
# skips runs whose best_model.pth already exists, so a session that dies midway
# resumes instead of starting over.
import subprocess
import time

epochs = 1 if SMOKE else EPOCHS
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for name in RUNS:
    spec = ABLATIONS[name]
    run_dir = OUTPUT_ROOT / name
    done = run_dir / spec["model"] / "best_model.pth"
    if done.exists():
        print(f"[skip] {name}: already trained ({done})")
        continue

    r = spec["recipe"]
    cmd = [
        "python", "scripts/train_kaggle.py",
        "--epochs", str(epochs),
        "--batch-size", str(BATCH_SIZE),
        "--seed", str(SEED),
        "--models", spec["model"],
        "--output-dir", str(run_dir),
        "--model-selection-metric", "val_f1_macro",
        "--optimizer", r["optimizer"],
        "--lr-scheduler", r["lr_scheduler"],
        "--label-smoothing", str(r["label_smoothing"]),
        "--loss", r["loss"],
        "--mixup-alpha", str(r["mixup_alpha"]),
        "--cutmix-alpha", str(r["cutmix_alpha"]),
        "--no-save-every-epoch",
    ]
    print(f"\n{'='*70}\n[run] {name}: {spec['question']}\n{' '.join(cmd)}\n{'='*70}")
    t0 = time.time()
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
    # Wall-clock is not logged by the training script; capture it here since the
    # thesis flags its absence as a limitation.
    mins = (time.time() - t0) / 60
    (run_dir / "runtime_minutes.txt").write_text(f"{mins:.2f}\n")

    # train_kaggle.py's summary.json records the model but not the recipe, so
    # without this an output directory cannot be traced back to the flags that
    # produced it - and which flags differed is the entire point of an ablation.
    import json as _json
    (run_dir / "ablation_config.json").write_text(_json.dumps({
        "run": name,
        "model": spec["model"],
        "question": spec["question"],
        "recipe": r,
        "epochs": epochs,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "smoke": SMOKE,
        "runtime_minutes": round(mins, 2),
    }, indent=2))
    print(f"[done] {name} in {mins:.1f} min")

## Source-conditioned analysis of each ablation

Macro F1 alone cannot answer the augmentation question - the claim is about the
**lab->field gap**, which only `source_analysis.csv` reports. This reruns each
ablation checkpoint over the same deterministic test split and writes the same
artifacts the thesis already quotes for the four main models.

In [ ]:
# --- Analyze every completed ablation (no retraining) -------------------------
# A smoke run trains for 1 epoch and scores near chance (1/76 = 0.013). Letting
# those numbers reach a CSV or a zip is how they end up quoted in the report by
# mistake, so this cell stops rather than producing a citable artifact.
if SMOKE:
    raise SystemExit(
        "SMOKE=True -> refusing to run: 1-epoch weights are not results. "
        "Set SMOKE=False in the config cell and retrain before analyzing."
    )

import subprocess

for name in RUNS:
    spec = ABLATIONS[name]
    run_dir = OUTPUT_ROOT / name
    if not (run_dir / spec["model"] / "best_model.pth").exists():
        print(f"[skip] {name}: not trained yet")
        continue
    cmd = [
        "python", "scripts/analyze_kaggle.py",
        "--model-root", str(run_dir),
        "--output-dir", str(run_dir),
        "--models", spec["model"],
        "--batch-size", str(BATCH_SIZE),
        "--seed", str(SEED),
    ]
    print(f"\n[analyze] {name}: {' '.join(cmd)}")
    subprocess.run(cmd, cwd=REPO_DIR, check=True)

## The attribution table

This is the output the thesis needs. It places each ablation between the two
endpoints already reported (v1 plain and v2 full) and attributes the gain.

Reference values from the thesis, for the two runs **not** repeated here:

| Model | Macro F1 | lab->field gap |
|---|---|---|
| Custom CNN (v1, plain) | 0.8735 | -17.5 pts |
| Custom CNN v2 (full) | 0.9633 | -6.7 pts |

If `aug_only` lands near -6.7 the augmentation claim is measured rather than
argued. If it stays near -17.5, the thesis claim is wrong and should be
retracted - which is a publishable finding either way.

In [ ]:
# --- Attribution table: macro F1 and lab->field gap for every completed run ---
# A smoke run trains for 1 epoch and scores near chance (1/76 = 0.013). Letting
# those numbers reach a CSV or a zip is how they end up quoted in the report by
# mistake, so this cell stops rather than producing a citable artifact.
if SMOKE:
    raise SystemExit(
        "SMOKE=True -> refusing to run: 1-epoch weights are not results. "
        "Set SMOKE=False in the config cell and retrain before analyzing."
    )

import csv
import json

# Endpoints already reported in the thesis (not retrained by this notebook).
REFERENCE = [
    ("custom_cnn (v1, plain)", "reported", 0.8735, -17.5),
    ("custom_cnn_v2 (full)", "reported", 0.9633, -6.7),
]


def read_run(run_dir, model):
    """Return (macro_f1, lab->field gap in points) for one analyzed run."""
    summary = run_dir / model / "analysis_summary.json"
    source = run_dir / model / "source_analysis.csv"
    if not summary.exists():
        return None, None
    macro = json.loads(summary.read_text()).get("macro_f1")
    gap = None
    if source.exists():
        acc = {}
        with open(source, newline="") as fh:
            for row in csv.DictReader(fh):
                acc[row["source"]] = float(row["accuracy"])
        if "leafsnap_lab" in acc and "leafsnap_field" in acc:
            gap = (acc["leafsnap_field"] - acc["leafsnap_lab"]) * 100
    return macro, gap


rows = list(REFERENCE)
for name in RUNS:
    spec = ABLATIONS[name]
    macro, gap = read_run(OUTPUT_ROOT / name, spec["model"])
    if macro is None:
        print(f"[pending] {name}: no analysis_summary.json yet")
        continue
    rows.append((f"{name} ({spec['model']})", "ablation", macro, gap))

print(f"\n{'run':34s} {'kind':10s} {'macro F1':>9s} {'lab->field':>11s}")
print("-" * 68)
for label, kind, macro, gap in rows:
    gap_s = f"{gap:+.1f} pts" if gap is not None else "n/a"
    print(f"{label:34s} {kind:10s} {macro:9.4f} {gap_s:>11s}")

if SMOKE:
    print("\n!! SMOKE=True -> these are 1-epoch numbers. Do NOT cite them.")

with open(OUTPUT_ROOT / "ablation_table.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["run", "kind", "macro_f1", "lab_to_field_gap_pts"])
    w.writerows(rows)
print(f"\nwrote {OUTPUT_ROOT / 'ablation_table.csv'}")

## Download bundle

In [ ]:
# --- Zip the metrics needed to write up the ablation --------------------------
# Weights are excluded: they are large and the report needs only the metrics.
# A smoke run trains for 1 epoch and scores near chance (1/76 = 0.013). Letting
# those numbers reach a CSV or a zip is how they end up quoted in the report by
# mistake, so this cell stops rather than producing a citable artifact.
if SMOKE:
    raise SystemExit(
        "SMOKE=True -> refusing to run: 1-epoch weights are not results. "
        "Set SMOKE=False in the config cell and retrain before analyzing."
    )

import zipfile

bundle = Path("/kaggle/working/ablation_bundle.zip")
keep = {".csv", ".json", ".png", ".txt"}
count = 0
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file() and path.suffix.lower() in keep:
            zf.write(path, path.relative_to(OUTPUT_ROOT.parent))
            count += 1

print(f"{bundle}  ({count} files, {bundle.stat().st_size/1e6:.1f} MB)")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file() and path.suffix.lower() in keep:
        print(" ", path.relative_to(OUTPUT_ROOT))